# Experiment D: Prior Sensitivity Analysis (§5.4)

Degradation parameter α ∈ [0, 1] blends the BIM prior toward uniform:
$$\pi_\alpha^{(e)} = (1-\alpha)\cdot\pi^{(e)} + \alpha\cdot u$$
At α=0: full prior; at α=1: baseline (no correction).

## Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# SDNET2018 sensitivity
with open('../data/sdnet6_sensitivity_results.json', 'r') as f:
    sdnet_sens = json.load(f)

# CODEBRIM sensitivity
with open('../data/experiment_C_results.json', 'r') as f:
    codebrim_sens = json.load(f)

alphas = [float(a) for a in sorted(sdnet_sens.keys(), key=float)]
print(f"Alpha values tested: {alphas}")

## SDNET2018 Results

In [ ]:
print('SDNET2018 Prior Sensitivity (EN 206-derived prior):')
print('=' * 65)
print(f"{'α':>5s} {'Accuracy':>10s} {'Macro F1':>10s} {'CE errors':>10s}")
print('-' * 65)
for a in alphas:
    d = sdnet_sens[str(a)]
    print(f"{a:>5.1f} {d['accuracy']:>10.4f} {d['macro_f1']:>10.4f} "
          f"{d['cross_element_errors']:>10d}")

print(f"\nRobustness: accuracy stable within ±0.05% up to α=0.5")
print(f"Cross-element errors: 4 at α=0, 24 at α=1 (monotonic increase)")

## CODEBRIM Results

In [ ]:
print('CODEBRIM Prior Sensitivity:')
print('=' * 55)
print(f"{'α':>5s} {'Accuracy':>10s} {'Macro F1':>10s}")
print('-' * 55)
for a in alphas:
    d = codebrim_sens[str(a)]
    print(f"{a:>5.1f} {d['accuracy']:>10.4f} {d['macro_f1']:>10.4f}")

## Figure 3 (Publication)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel (a): SDNET2018
acc_s = [sdnet_sens[str(a)]['accuracy'] for a in alphas]
f1_s = [sdnet_sens[str(a)]['macro_f1'] for a in alphas]
ce_s = [sdnet_sens[str(a)]['cross_element_errors'] for a in alphas]

ax1.plot(alphas, acc_s, 'o-', color='#2196F3', label='Accuracy', linewidth=2)
ax1.plot(alphas, f1_s, 's-', color='#FF9800', label='Macro F1', linewidth=2)
ax1.set_xlabel('α (prior degradation)', fontsize=11)
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('(a) SDNET2018', fontsize=12)
ax1.legend(loc='lower left')
ax1.grid(alpha=0.3)

ax1r = ax1.twinx()
ax1r.bar(alphas, ce_s, width=0.06, alpha=0.3, color='#F44336', label='CE errors')
ax1r.set_ylabel('Cross-element errors', color='#F44336')
ax1r.legend(loc='upper right')

# Panel (b): CODEBRIM
acc_c = [codebrim_sens[str(a)]['accuracy'] for a in alphas]
f1_c = [codebrim_sens[str(a)]['macro_f1'] for a in alphas]

ax2.plot(alphas, acc_c, 'o-', color='#2196F3', label='Accuracy', linewidth=2)
ax2.plot(alphas, f1_c, 's-', color='#FF9800', label='Macro F1', linewidth=2)
ax2.set_xlabel('α (prior degradation)', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('(b) CODEBRIM', fontsize=12)
ax2.legend(loc='lower left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/nb04_prior_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Per-class F1 Sensitivity (SDNET2018)

In [ ]:
class_names = list(sdnet_sens['0.0']['per_class_f1'].keys())
fig, ax = plt.subplots(figsize=(10, 5))
for cls in class_names:
    vals = [sdnet_sens[str(a)]['per_class_f1'][cls] for a in alphas]
    ax.plot(alphas, vals, 'o-', label=cls, linewidth=1.5)
ax.set_xlabel('α')
ax.set_ylabel('F1')
ax.set_title('Per-class F1 vs α (SDNET2018)')
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nb04_perclass_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- SDNET2018: BIM-BPC robust to prior degradation up to α ≈ 0.5 (< 0.1% accuracy loss).
- Cross-element errors increase monotonically with α (4 → 24).
- CODEBRIM: modest improvement at α=0, performance stable across range.
- Conclusion: even a partially correct prior provides meaningful correction.